In [ ]:
# cell 1: imports + small utilities

import numpy as np
import matplotlib.pyplot as plt

from dataclasses import dataclass
from typing import Dict, Tuple

def trapz_int(y: np.ndarray, x: np.ndarray, axis: int = -1) -> np.ndarray:
    """simple trapezoid integral wrapper"""
    return np.trapz(y, x, axis=axis)

def gaussian(x, mu, sig):
    """unit-area gaussian in x"""
    return np.exp(-0.5*((x - mu)/sig)**2) / (sig*np.sqrt(2*np.pi))

def safe_div(a, b, eps=1e-30):
    """avoid divide by zero"""
    return a / (b + eps)

In [ ]:
# cell 2: configuration object (edit these)

@dataclass
class SimConfig:
    # wavelength band (meters)
    lam_min: float = 350e-9
    lam_max: float = 1100e-9
    n_lam: int = 8000  # internal grid for forward model integration

    # mkid coarse energy resolution (used for binning + order-edge ambiguity model)
    mkid_R: float = 40.0  # you said ~35–50

    # ifts scan geometry
    opd_max: float = 1.0e-3      # meters of opd (edit based on desired resolution)
    n_steps_total: int = 2500    # total opd samples (time budget constraint)
    frac_negative: float = 0.10  # fraction of points on negative side (phase side)
    # note: negative side extent uses the same opd_max scaling below

    # integration time
    t_step: float = 2.0          # seconds per opd step (exposure time per frame)
    overhead_step: float = 0.0   # seconds overhead per step (set nonzero if needed)

    # telescope / instrument geometry
    telescope_diam: float = 3.6  # meters
    pix_solid_angle: float = 1.0 # placeholder sr per detector pixel (set later)

    # output selection
    use_two_port: bool = True    # recommended for proper normalization

cfg = SimConfig()

In [ ]:
# cell 3: wavelength grid + example "truth" target + sky (replace with real spectra)

lam = np.linspace(cfg.lam_min, cfg.lam_max, cfg.n_lam)  # meters
nu = 3e8 / lam                                          # hz (only if you need it)
sigma = 1.0 / lam                                       # 1/m (wavenumber for ifts math)

# example target: flat continuum + a gaussian emission line
# units here are "photons / s / m^2 / m" entering the atmosphere (arbitrary for now)
cont = 1.0e12 * np.ones_like(lam)
line = 5.0e12 * gaussian(lam, mu=656.3e-9, sig=0.35e-9)
target_in = cont + line

# example sky background: smooth + weak structure
sky_in = 2.0e12 * (lam / lam.mean())**(-0.8)

plt.figure()
plt.plot(lam*1e9, target_in, label="target_in")
plt.plot(lam*1e9, sky_in, label="sky_in")
plt.xlabel("wavelength (nm)")
plt.ylabel("photon spectral density (arb)")
plt.legend()
plt.show()

In [ ]:
# cell 4: throughput model blocks (simple placeholders you can swap with real curves)

def atmosphere_transmission(lam_m: np.ndarray, airmass: float = 1.2) -> np.ndarray:
    # placeholder: gentle slope + a weak absorption band
    base = 0.92 - 0.08*(lam_m - lam_m.min())/(lam_m.max() - lam_m.min())
    band = 1.0 - 0.08*np.exp(-0.5*((lam_m - 760e-9)/(18e-9))**2)  # fake o2-like dip
    t = np.clip(base*band, 0.0, 1.0)
    # crude airmass scaling (replace with real model later)
    return np.clip(t**airmass, 0.0, 1.0)

def telescope_throughput(lam_m: np.ndarray, n_mirrors: int = 3, r_mirror: float = 0.90) -> np.ndarray:
    # simple: r^n
    return np.clip(r_mirror**n_mirrors, 0.0, 1.0) * np.ones_like(lam_m)

def ifts_optics_throughput(lam_m: np.ndarray) -> np.ndarray:
    # placeholder: mild chromatic response
    t = 0.70 + 0.10*np.cos(2*np.pi*(lam_m - lam_m.min())/(lam_m.max() - lam_m.min()))
    return np.clip(t, 0.0, 1.0)

def mkid_qe(lam_m: np.ndarray) -> np.ndarray:
    # placeholder: peaked qe
    qe = 0.65*np.exp(-0.5*((lam_m - 700e-9)/(260e-9))**2) + 0.10
    return np.clip(qe, 0.0, 1.0)

# total throughput (multiplicative)
T_atm = atmosphere_transmission(lam)
T_tel = telescope_throughput(lam)
T_ifts = ifts_optics_throughput(lam)
QE = mkid_qe(lam)

T_total = T_atm * T_tel * T_ifts * QE

target_det = target_in * T_total
sky_det = sky_in * T_total

plt.figure()
plt.plot(lam*1e9, T_total, label="T_total")
plt.xlabel("wavelength (nm)")
plt.ylabel("throughput (arb)")
plt.legend()
plt.show()

In [ ]:
# cell 5: opd grid (asymmetric scan) + resolution helpers

def make_opd_grid(opd_max: float, n_total: int, frac_negative: float) -> np.ndarray:
    # negative side: short scan for phase estimation
    n_neg = max(8, int(np.floor(n_total * frac_negative)))
    n_pos = n_total - n_neg

    # choose negative extent as a fraction of opd_max
    # edit this if your prof wants a different negative distance rule
    opd_neg_max = -0.25 * opd_max  # short negative reach (phase side)
    opd_pos_max = +1.00 * opd_max

    opd_neg = np.linspace(opd_neg_max, 0.0, n_neg, endpoint=False)
    opd_pos = np.linspace(0.0, opd_pos_max, n_pos, endpoint=True)

    return np.concatenate([opd_neg, opd_pos])

opd = make_opd_grid(cfg.opd_max, cfg.n_steps_total, cfg.frac_negative)

# simple resolution proxy: delta_sigma ~ 1/(2*opd_max), in wavenumber
delta_sigma = 1.0 / (2.0 * cfg.opd_max)  # 1/m

plt.figure()
plt.plot(opd*1e3, np.zeros_like(opd), ".", ms=2)
plt.xlabel("opd (mm)")
plt.yticks([])
plt.title("opd sampling (asymmetric)")
plt.show()

print("n_steps_total =", len(opd))
print("delta_sigma ~", delta_sigma, "1/m")

In [ ]:
# cell 6: mkid "order bins" + order-edge ambiguity map (simple, but useful)

def mkid_bins_from_R(lam_m: np.ndarray, R: float) -> Tuple[np.ndarray, np.ndarray]:
    # build bins with width dlam ~ lam/R (constant resolving power)
    # we create bin edges by stepping in log-lambda
    loglam = np.log(lam_m)
    dlog = np.mean(np.diff(loglam))
    # target bin width in log space: dlam/lam = 1/R -> dlog ~ 1/R
    dlog_bin = 1.0 / R
    n_bins = max(4, int(np.floor((loglam[-1] - loglam[0]) / dlog_bin)))
    edges = np.linspace(loglam[0], loglam[-1], n_bins + 1)
    lam_edges = np.exp(edges)
    lam_centers = np.sqrt(lam_edges[:-1] * lam_edges[1:])
    return lam_edges, lam_centers

lam_edges, lam_centers = mkid_bins_from_R(lam, cfg.mkid_R)

# ambiguity model: if a wavelength sits within 1 sigma of a bin edge, mark as "edge-risk"
# sigma_edge uses mkid energy resolution as gaussian blur in lambda-space: sig ~ lam/(2.355*R)
sig_lam = lam / (2.355 * cfg.mkid_R)

edge_risk = np.zeros_like(lam, dtype=float)
for e in lam_edges[1:-1]:
    edge_risk = np.maximum(edge_risk, np.exp(-0.5*((lam - e)/sig_lam)**2))

plt.figure()
plt.plot(lam*1e9, edge_risk)
plt.xlabel("wavelength (nm)")
plt.ylabel("order-edge risk (0..1)")
plt.title("mkid order-edge ambiguity indicator (simple)")
plt.show()

print("n_mkid_bins =", len(lam_centers))

In [ ]:
# cell 7: ifts forward model (2-port interferograms from spectra)
# note: this is the core physical forward model (no extra 0.5 factor)

def interferogram_two_port(opd: np.ndarray, sigma: np.ndarray, S_sigma: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """
    build ideal 2-port interferograms:
    i_plus(opd)  = integral S(sigma) * (1 + cos(2*pi*sigma*opd)) d sigma
    i_minus(opd) = integral S(sigma) * (1 - cos(2*pi*sigma*opd)) d sigma
    """
    # cos matrix: shape (n_opd, n_sigma)
    cos_mat = np.cos(2.0*np.pi * opd[:, None] * sigma[None, :])

    # integrate over sigma
    dc = trapz_int(S_sigma[None, :] * np.ones_like(cos_mat), sigma, axis=1)
    ac = trapz_int(S_sigma[None, :] * cos_mat, sigma, axis=1)

    i_plus = dc + ac
    i_minus = dc - ac
    return i_plus, i_minus

# convert from per-lambda spectral density to per-sigma spectral density:
# s_sigma d sigma = s_lam d lam, and sigma = 1/lam => d sigma = - d lam / lam^2
# so s_sigma = s_lam * |d lam / d sigma| = s_lam * lam^2
target_sigma = target_det * (lam**2)
sky_sigma = sky_det * (lam**2)

# total at detector = target + sky (you can keep separate if you prefer)
total_sigma = target_sigma + sky_sigma

i_plus, i_minus = interferogram_two_port(opd, sigma, total_sigma)

plt.figure()
plt.plot(opd*1e3, i_plus, label="i_plus")
plt.plot(opd*1e3, i_minus, label="i_minus", alpha=0.8)
plt.xlabel("opd (mm)")
plt.ylabel("interferogram signal (arb)")
plt.legend()
plt.show()

In [ ]:
# cell 8: add noise (poisson) + reconstruct spectrum by fft (simple v1)

def add_poisson_noise(interf: np.ndarray, t_step: float, overhead: float = 0.0, rng=None) -> Tuple[np.ndarray, np.ndarray]:
    # treat interferogram value as photon rate (arb), convert to expected counts per step
    # if you later calibrate real units, this becomes physically meaningful
    if rng is None:
        rng = np.random.default_rng(0)

    t_eff = t_step  # you can incorporate overhead differently if needed
    mu = np.clip(interf * t_eff, 0.0, None)
    counts = rng.poisson(mu)
    noisy_rate = counts / t_eff
    var_rate = np.clip(mu, 0.0, None) / (t_eff**2)  # var(poisson)=mu
    return noisy_rate, var_rate

# choose a single "science interferogram" stream
# common choice: difference port removes dc term and emphasizes modulation
if cfg.use_two_port:
    interf_clean = 0.5*(i_plus - i_minus)  # equals ac term (no double counting)
else:
    interf_clean = i_plus  # fallback

noisy_interf, var_interf = add_poisson_noise(interf_clean, cfg.t_step, cfg.overhead_step)

plt.figure()
plt.plot(opd*1e3, interf_clean, label="clean")
plt.plot(opd*1e3, noisy_interf, label="noisy", alpha=0.7)
plt.xlabel("opd (mm)")
plt.ylabel("modulated interferogram (arb)")
plt.legend()
plt.show()

def reconstruct_spectrum_fft(opd: np.ndarray, interf: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """
    v1 reconstruction:
    - assumes opd is uniform enough for np.fft (approx)
    - returns sigma_axis (1/m) and spectrum (arb)
    if opd is non-uniform (asymmetric spacing differences), you will want a nufft later.
    """
    # check spacing
    d = np.median(np.diff(opd))
    n = len(opd)

    # window (optional): use a mild apodization to suppress sidelobes
    w = np.hanning(n)
    x = (interf - np.mean(interf)) * w

    spec = np.fft.rfft(x)
    freq = np.fft.rfftfreq(n, d=d)  # cycles per meter, i.e. sigma (1/m)
    # magnitude as a proxy for spectral density (v1)
    return freq, np.abs(spec)

sigma_rec, spec_rec = reconstruct_spectrum_fft(opd, noisy_interf)

plt.figure()
plt.plot(sigma_rec, spec_rec)
plt.xlabel("wavenumber sigma (1/m)")
plt.ylabel("reconstructed spectrum (arb)")
plt.title("v1 fft reconstruction (magnitude)")
plt.show()

# map reconstructed sigma to wavelength for plotting
lam_rec = safe_div(1.0, sigma_rec)
mask = (lam_rec >= cfg.lam_min) & (lam_rec <= cfg.lam_max)

plt.figure()
plt.plot(lam_rec[mask]*1e9, spec_rec[mask])
plt.xlabel("wavelength (nm)")
plt.ylabel("reconstructed spectrum (arb)")
plt.title("reconstructed spectrum over band")
plt.gca().invert_xaxis()  # optional: wavelength decreases with sigma
plt.show()

In [ ]:
# cell 9: snr(λ) estimator (v1) + edge-risk overlay

def snr_proxy_from_mc(opd, sigma, total_sigma, n_mc=50, seed=1):
    # quick monte-carlo to estimate variance after reconstruction
    rng = np.random.default_rng(seed)

    if cfg.use_two_port:
        clean = 0.5*(i_plus - i_minus)
    else:
        clean = i_plus

    sigma_axis, spec_mean = None, None
    specs = []

    for _ in range(n_mc):
        noisy, _ = add_poisson_noise(clean, cfg.t_step, cfg.overhead_step, rng=rng)
        s_ax, s_rec = reconstruct_spectrum_fft(opd, noisy)
        specs.append(s_rec)
        sigma_axis = s_ax

    specs = np.array(specs)
    mean = specs.mean(axis=0)
    std = specs.std(axis=0, ddof=1)
    return sigma_axis, mean, std

sigma_mc, mean_mc, std_mc = snr_proxy_from_mc(opd, sigma, total_sigma, n_mc=30, seed=2)
snr_mc = safe_div(mean_mc, std_mc)

lam_mc = safe_div(1.0, sigma_mc)
mask = (lam_mc >= cfg.lam_min) & (lam_mc <= cfg.lam_max)

plt.figure()
plt.plot(lam_mc[mask]*1e9, snr_mc[mask], label="snr proxy (mc)")
# overlay edge-risk (resampled) so gaps show up naturally
edge_interp = np.interp(lam_mc[mask], lam, edge_risk)
plt.plot(lam_mc[mask]*1e9, 0.2*np.max(snr_mc[mask])*edge_interp, label="scaled order-edge risk", alpha=0.7)
plt.xlabel("wavelength (nm)")
plt.ylabel("snr (arb)")
plt.title("snr proxy with mkid order-edge ambiguity overlay")
plt.legend()
plt.gca().invert_xaxis()
plt.show()